In [ ]:
# pandas: tabular data loading and column manipulation
# numpy: numerical operations used implicitly by scipy and math downstream
# scipy.stats: provides percentileofscore() for computing per-player percentile ranks
#   relative to the full population of defenders in the dataset
# math: provides floor() to convert float percentiles to integers required by PyPizza
# mplsoccer.PyPizza: renders the pizza/donut-style percentile radar chart
# mplsoccer.add_image: utility to overlay images (e.g., club crests) on the chart canvas
# mplsoccer.FontManager: downloads and registers custom web fonts for chart typography
# matplotlib.pyplot: base figure management and file export
import pandas as pd
import numpy as np
from scipy import stats
import math
from mplsoccer import PyPizza, add_image, FontManager
import matplotlib.pyplot as plt

# Load the FBref player stats CSV for the 2020-21 Premier League season.
# FBref exports player names in a backslash-separated format:
# "Full Name\url-slug" (e.g., "Trent Alexander-Arnold\Trent-Alexander-Arnold").
# str.split('\\', expand=True)[0] splits on the literal backslash and retains only
# the first token — the human-readable full name — for clean filtering downstream.
df = pd.read_csv("data/pizza_tutorial9.csv")
df['Player'] = df['Player'].str.split('\\', expand=True)[0]
df.head()

In [ ]:
# Filter to defenders who played at least 15 full 90-minute equivalents.
# df['Pos'] == 'DF' restricts to outfield defensive positions only, excluding
# midfielders and forwards who may be listed with a secondary position.
# df['90s'] > 15 enforces a minimum playing time threshold — approximately 1,350 minutes.
# Computing percentile ranks against players with very few appearances produces
# unreliable scores: a defender who played 3 games and had one fortunate match
# would sit at the 90th percentile for a metric, which is not representative.
df = df.loc[(df['Pos'] == 'DF') & (df['90s'] > 15)]
df.head()

In [ ]:
# Drop administrative and demographic columns that are not performance metrics.
# Keeping them in the DataFrame would include them in the params list and pollute
# the pizza chart with meaningless axes (e.g., a "Born year" slice).
# reset_index() resets the integer row index after the positional filter applied above,
# ensuring iloc[] and .at[] calls work correctly in subsequent cells.
df = df.drop(['Rk', 'Nation', 'Pos', 'Squad', 'Age', 'Born'], axis=1).reset_index()
df.head()

In [ ]:
# Extract the metric column names to use as pizza chart axis labels.
# list(df.columns) converts the Index to a plain Python list for slicing.
# params[2:] skips the first two columns ('index' and 'Player'), which are row
# identifiers — not performance metrics. The remaining 14 columns are the
# defensive stats that will each occupy one slice of the pizza chart.
params = list(df.columns)
params = params[2:]
params

In [ ]:
# Isolate the target player's row and convert it to a plain Python list.
# .reset_index() is called on the filtered DataFrame before .loc[0] to ensure
# the first (and only) matching row is accessible at index position 0.
# list(player.loc[0]) converts the row to a list in column order:
# [level_0, index, 'Player name', stat_1, stat_2, ..., stat_n]
# The two index columns and player name are trimmed in the next cell.
player = df.loc[df['Player'] == 'Trent Alexander-Arnold'].reset_index()
player = list(player.loc[0])
print(player)

In [46]:
df.Player.values

array(['Patrick van Aanholt', 'Tosin Adarabioyo', 'Ola Aina',
       'Semi Ajayi', 'Toby Alderweireld', 'Trent Alexander-Arnold',
       'Ezgjan Alioski', 'Joachim Andersen', 'Serge Aurier',
       'Luke Ayling', 'César Azpilicueta', 'George Baldock',
       'Kyle Bartley', 'Jan Bednarek', 'Héctor Bellerín', 'Ryan Bertrand',
       'Willy Boly', 'Dan Burn', 'Gary Cahill', 'João Cancelo',
       'Matty Cash', 'Timothy Castagne', 'Ben Chilwell',
       'Andreas Christensen', 'Ciaran Clark', 'Conor Coady',
       'Séamus Coleman', 'Liam Cooper', 'Vladimír Coufal',
       'Aaron Cresswell', 'Craig Dawson', 'Rúben Dias', 'Eric Dier',
       'Lucas Digne', 'Issa Diop', 'Gabriel Dos Santos', 'Lewis Dunk',
       'John Egan', 'Jonny Evans', 'Federico Fernández', 'Wesley Fofana',
       'Darnell Furlong', 'Ben Godfrey', 'Rob Holding', 'Mason Holgate',
       'Reece James', 'James Justin', 'Michael Keane', 'Ezri Konsa',
       'Cheikhou Kouyaté', 'Jamaal Lascelles', 'Jamal Lewis',
       'Victor

In [ ]:
# Trim the player list to remove the two index prefixes and the Player name string,
# leaving only the numeric stat values in the same order as the params list.
# player[3:] skips 'level_0', 'index', and 'Player' — the first three elements.
player = player[3:]

values = []
for x in range(len(params)):
    # stats.percentileofscore(array, score) returns the percentile rank of `score`
    # within the distribution of all defenders in df[params[x]].
    # A score of 80 means the player outperforms 80% of all defenders in that metric.
    # math.floor() converts the float result to an integer as required by PyPizza's input.
    values.append(math.floor(stats.percentileofscore(df[params[x]], player[x])))

print(round(stats.percentileofscore(df[params[0]], player[0])))

In [ ]:
# PyPizza requires all slice values to be in the range [0, 99].
# A raw percentile of 100 (player is better than every other player in the sample)
# causes PyPizza to render a fully closed slice that wraps around the outer ring,
# which breaks the visual layout. Capping at 99 preserves the intent (best in group)
# while remaining within the accepted input range.
for n, i in enumerate(values):
    if i == 100:
        values[n] = 99

In [ ]:
# PyPizza() instantiates the pizza chart renderer with global layout settings.
#   params: list of metric names displayed as axis labels around the chart perimeter
#   straight_line_color/lw: color and width of the radial dividers separating each slice
#   last_circle_lw: line width of the outermost ring (representing the 99th percentile)
#   other_circle_lw/ls: line width and style of inner reference rings (25th, 50th, 75th)
baker = PyPizza(
    params=params,
    straight_line_color='#000000',
    straight_line_lw=1,
    last_circle_lw=1,
    other_circle_lw=1,
    other_circle_ls='-'
)

# baker.make_pizza() renders the full chart and returns the figure and axes.
#   values: list of integer percentile scores (0–99) — one per param
#   param_location=110: positions axis labels at 110% of the outer ring radius
#   kwargs_slices: styling for each filled pizza slice (facecolor, edge, z-order)
#   kwargs_params: styling for the metric name labels at the perimeter
#   kwargs_values: styling for the percentile number annotations on each slice;
#     bbox defines the rounded rectangle drawn behind each number for readability
fig, ax = baker.make_pizza(
    values,
    figsize=(8, 8),
    param_location=110,
    kwargs_slices=dict(
        facecolor="#6CABDD", edgecolor="#000000",
        zorder=2, linewidth=1
    ),
    kwargs_params=dict(
        color="#000000", fontsize=12,
        va="center", alpha=.5
    ),
    kwargs_values=dict(
        color="#000000", fontsize=12,
        zorder=3,
        bbox=dict(
            edgecolor="#000000", facecolor="#6CABDD",
            boxstyle="round,pad=0.2", lw=1
        )
    )
)

# Title and subtitle use figure-relative coordinates (0–1).
# ha="center" on x=0.515 (slightly right of center) visually centers text on the chart.
fig.text(0.515, 0.97, "Trent Alexander-Arnold - Liverpool", size=18,
         ha="center", color="#000000")
fig.text(
    0.515, 0.942,
    "Percentile vs all Premier League defenders with 15+ 90s played | 2020-21",
    size=15, ha="center", color="#000000"
)

notes = 'Players with more than 15 full 90-minute equivalents played'
CREDIT_1 = "Data: FBref"
CREDIT_2 = 'Inspired by futboldata_academy'

# Bottom-right footnote with ha="right" anchored at (0.99, 0.005).
fig.text(0.99, 0.005, f"{notes}\n{CREDIT_1}\n{CREDIT_2}", size=9,
         color="#000000", ha="right")

# Export the chart as a high-resolution PNG.
# dpi=500 produces a print-quality image; bbox_inches='tight' trims excess whitespace.
plt.savefig('pizza.png', dpi=500, bbox_inches='tight')

## Summary: Pizza Radar Chart — Single Player Percentile Profile

### What This Notebook Does

This notebook constructs a pizza/donut-style radar chart that shows a single Premier League defender (Trent Alexander-Arnold) ranked as a percentile across 14 defensive metrics relative to all defenders in the 2020-21 season who played at least 15 full 90-minute equivalents. The pizza chart is a widely used format in modern football analytics media for communicating a player's all-around profile at a glance.

### Key Concepts

- **FBref name format**: FBref exports player names as `"Full Name\url-slug"`. The `str.split('\\', expand=True)[0]` pattern is a reusable cleaning step applicable to any FBref dataset — it splits on the literal backslash character (escaped as `\\` in Python) and discards the URL component.
- **Minimum playing time filter** (`90s > 15`): Applying a sample size threshold before computing percentiles is standard practice in football analytics. Without it, a player with two outstanding games in a small sample would rank at the 95th percentile, which is misleading when compared to players with 30+ appearances.
- **`stats.percentileofscore(array, value)`**: For each metric, this function computes what fraction of the entire defender population scores below the target player. The result is a number between 0 and 100 representing the player's relative ranking in that metric — not their raw stat, but where they stand among peers.
- **`math.floor()` vs `round()`**: `floor()` is used instead of `round()` to be conservative — a 79.9th percentile is shown as 79, not 80. This prevents slight overstatement of the player's position in any metric.
- **99-cap for PyPizza**: PyPizza's rendering engine reserves the value 100 for the outer boundary of the chart, not for data. Any player who ranks at the absolute top of a metric must be capped at 99 to avoid a rendering artifact where the slice closes and overflows the outer ring.
- **`plt.savefig(dpi=500)`**: High DPI export is important for pizza charts used in publications or social media — at 96 DPI the axis labels and value boxes become pixelated.

### Data Available

| Object | Description |
|---|---|
| `df` | Filtered DataFrame of Premier League defenders with 15+ 90s, 14 metric columns |
| `params` | List of 14 metric column names used as pizza chart axis labels |
| `player` | Numeric stat list for Trent Alexander-Arnold (in `params` order) |
| `values` | Integer percentile scores (0–99) for each metric — direct input to PyPizza |

### Ideas to Extract More Value

- **Multi-player comparison**: Generate pizza charts for multiple players and display them in a grid (`plt.subplots`) to visually compare defensive profiles across a squad or across positional peers.
- **Position-specific percentiles**: The current comparison pool is all defenders. Splitting further into centre-backs vs full-backs produces more meaningful within-role rankings — Alexander-Arnold's pressing metrics look very different when compared to full-backs only vs all defenders.
- **Color-coded performance zones**: Use three colors for the pizza slices — red for below 33rd percentile, amber for 33–66th, green for 66th+ — to immediately highlight strengths and weaknesses without reading individual numbers.
- **Season-over-season tracking**: Load FBref data for multiple seasons and compute pizza charts for each. Stacking or animating them reveals how a player's profile evolves — useful for development monitoring or transfer scouting.
- **Automated reporting pipeline**: Wrap the full pipeline (load, filter, compute percentiles, render) into a function `generate_pizza(player_name, season_df)` that accepts any player name and returns the chart, enabling batch generation for full squad reports.